# 🎙️ OpenBMB VoxCPM2 Voice Cloning & LoRA Fine-Tuning
### Optimized for Google Colab (T4 16GB GPU / A100 / V100)

This self-contained notebook trains a LoRA adapter on `openbmb/VoxCPM2` with low-VRAM optimizations:
- **Batch Size**: 2 per GPU
- **Gradient Accumulation**: 8 steps (effective batch size: 16)
- **Max Batch Tokens**: 2048
- **LoRA Rank (r)**: 32 | **LoRA Alpha**: 64
- **Audio VAE Input**: 16,000 Hz | **Audio Output**: 48,000 Hz
- **Epochs**: 2
- Auto-zips and downloads the final LoRA checkpoint.

## 1. Inspect GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/OpenBMB/VoxCPM.git
%cd VoxCPM
!pip install --upgrade pip
!pip install -e .
!pip install soundfile huggingface_hub pyyaml accelerate peft datasets

## 3. Prepare Training Data
Upload your `dataset.zip` (containing your `.wav` files) and `train.jsonl` + `val.jsonl`.
If you used `prepare_dataset.py` locally, you can upload its outputs here.

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('dataset', exist_ok=True)
print('Upload your dataset.zip (audio files) and train.jsonl / val.jsonl:')
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print(f'Extracted {filename} into ./dataset')
    elif filename.endswith('.jsonl'):
        print(f'Uploaded manifest: {filename}')


## 4. Configure Low-VRAM LoRA Hyperparameters
Writes the fine-tuning configuration file `conf/voxcpm_v2/voxcpm_finetune_lora_lowvram.yaml`.

In [ ]:
import yaml
from pathlib import Path

config = {
    'pretrained_path': 'openbmb/VoxCPM2',
    'train_manifest': 'train.jsonl',
    'val_manifest': 'val.jsonl' if Path('val.jsonl').exists() else 'train.jsonl',
    'output_dir': './output_lora',
    'learning_rate': 1e-4,
    'batch_size': 2,
    'grad_accum_steps': 8,
    'max_batch_tokens': 2048,
    'num_epochs': 2,
    'sample_rate': 16000,
    'out_sample_rate': 48000,
    'lora': {
        'r': 32,
        'alpha': 64,
        'dropout': 0.05,
        'enable_lm': True,
        'enable_dit': True,
        'enable_proj': False,
    },
    'optim': {
        'optimizer': 'adamw',
        'weight_decay': 0.01,
        'max_grad_norm': 1.0,
        'warmup_steps': 50,
    },
    'save_interval_steps': 100,
    'logging_interval_steps': 10,
}

cfg_path = Path('conf/voxcpm_v2/voxcpm_finetune_lora_lowvram.yaml')
cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, sort_keys=False)

print(f'Saved low-VRAM training config to: {cfg_path}')


## 5. Execute VoxCPM2 LoRA Fine-Tuning
Runs distributed training via PyTorch Accelerate.

In [ ]:
!accelerate launch src/voxcpm/training/train.py --config conf/voxcpm_v2/voxcpm_finetune_lora_lowvram.yaml

## 6. Test Fine-Tuned Voice Generation
Load your trained LoRA adapter directly into VoxCPM2 and test voice synthesis at 48 kHz.

In [ ]:
from voxcpm import VoxCPM
import soundfile as sf
from IPython.display import Audio, display

# Load model with trained LoRA weights
model = VoxCPM.from_pretrained('openbmb/VoxCPM2', lora_weights_path='./output_lora', device='cuda')

prompt_text = 'Hello, this is a demonstration of our custom fine-tuned VoxCPM2 voice model.'
audio = model.generate(text=prompt_text, inference_timesteps=8, cfg_value=2.0)

sf.write('test_generation.wav', audio, 48000)
print('Synthesis complete! Playing sample:')
display(Audio('test_generation.wav'))


## 7. Package & Download LoRA Checkpoint
Zips `./output_lora` so you can use it locally in Vishnu Creator or `clone_cli.py` / `app.py`.

In [ ]:
!zip -r voxcpm2_lora_checkpoint.zip ./output_lora
files.download('voxcpm2_lora_checkpoint.zip')
print('Download initiated: voxcpm2_lora_checkpoint.zip')
